	比较模型，交叉验证选参，保存最终模型

In [1]:
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error

component_columns = ["Aryl halide", "Ligand", "Base", "Additive"]

df = pd.read_excel(
    "data/raw/Dreher_and_Doyle_input_data.xlsx",
    sheet_name="FullCV_01",
    engine="openpyxl",
)

splits = pd.read_csv(
    "data/splits/split_manifest.csv",
    index_col="source_row_index",
)

assert df.index.equals(splits.index)
assert df[component_columns].equals(splits[component_columns])

random_train = df.loc[splits["random_split"].eq("train")]
random_test = df.loc[splits["random_split"].eq("test")]

print("Data and split manifest match.")
print("Training records:", len(random_train))
print("Test records:", len(random_test))

Data and split manifest match.
Training records: 2768
Test records: 1187


In [2]:
mean_baseline = DummyRegressor(strategy="mean")

mean_baseline.fit(
    random_train[component_columns],
    random_train["Output"],
)

baseline_predictions = mean_baseline.predict(
    random_test[component_columns]
)

baseline_mae = mean_absolute_error(
    random_test["Output"],
    baseline_predictions,
)

print(f"Training mean yield: {random_train['Output'].mean():.2f}%")
print("First five predictions:", np.round(baseline_predictions[:5], 2))
print(f"Test MAE: {baseline_mae:.2f} percentage points")

Training mean yield: 32.63%
First five predictions: [32.63 32.63 32.63 32.63 32.63]
Test MAE: 23.65 percentage points


In [3]:
additive_train = df.loc[splits["additive_split"].eq("train")]
additive_test = df.loc[splits["additive_split"].eq("test")]

additive_mean_baseline = DummyRegressor(strategy="mean")

additive_mean_baseline.fit(
    additive_train[component_columns],
    additive_train["Output"],
)

additive_baseline_predictions = additive_mean_baseline.predict(
    additive_test[component_columns]
)

additive_baseline_mae = mean_absolute_error(
    additive_test["Output"],
    additive_baseline_predictions,
)

print("Training records:", len(additive_train))
print("Test records:", len(additive_test))
print(f"Training mean yield: {additive_train['Output'].mean():.2f}%")
print(f"Test MAE: {additive_baseline_mae:.2f} percentage points")

Training records: 3775
Test records: 180
Training mean yield: 33.15%
Test MAE: 23.36 percentage points


In [4]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

random_linear_model = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ("regressor", LinearRegression()),
])

random_linear_model.fit(
    random_train[component_columns],
    random_train["Output"],
)

random_linear_predictions = random_linear_model.predict(
    random_test[component_columns]
)

random_linear_mae = mean_absolute_error(
    random_test["Output"],
    random_linear_predictions,
)

print("First five predictions:", np.round(random_linear_predictions[:5], 2))
print(f"Mean baseline MAE: {baseline_mae:.2f} percentage points")
print(f"Linear regression MAE: {random_linear_mae:.2f} percentage points")

First five predictions: [34.62  1.88 20.15  4.17 39.48]
Mean baseline MAE: 23.65 percentage points
Linear regression MAE: 12.25 percentage points


In [5]:
additive_linear_model = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ("regressor", LinearRegression()),
])

additive_linear_model.fit(
    additive_train[component_columns],
    additive_train["Output"],
)

additive_linear_predictions = additive_linear_model.predict(
    additive_test[component_columns]
)

additive_linear_mae = mean_absolute_error(
    additive_test["Output"],
    additive_linear_predictions,
)

print("First five predictions:", np.round(additive_linear_predictions[:5], 2))
print(f"Mean baseline MAE: {additive_baseline_mae:.2f} percentage points")
print(f"Linear regression MAE: {additive_linear_mae:.2f} percentage points")

First five predictions: [ 74.38  11.93  54.57  41.8  -14.53]
Mean baseline MAE: 23.36 percentage points
Linear regression MAE: 13.73 percentage points


In [6]:
from sklearn.ensemble import RandomForestRegressor

random_forest_model = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    )),
])

random_forest_model.fit(
    random_train[component_columns],
    random_train["Output"],
)

random_forest_predictions = random_forest_model.predict(
    random_test[component_columns]
)

random_forest_mae = mean_absolute_error(
    random_test["Output"],
    random_forest_predictions,
)

print("First five predictions:", np.round(random_forest_predictions[:5], 2))
print(f"Mean baseline MAE: {baseline_mae:.2f} percentage points")
print(f"Linear regression MAE: {random_linear_mae:.2f} percentage points")
print(f"Random forest MAE: {random_forest_mae:.2f} percentage points")

First five predictions: [11.54 15.93  0.45  0.35 31.01]
Mean baseline MAE: 23.65 percentage points
Linear regression MAE: 12.25 percentage points
Random forest MAE: 6.03 percentage points


In [7]:
additive_forest_model = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    )),
])

additive_forest_model.fit(
    additive_train[component_columns],
    additive_train["Output"],
)

additive_forest_predictions = additive_forest_model.predict(
    additive_test[component_columns]
)

additive_forest_mae = mean_absolute_error(
    additive_test["Output"],
    additive_forest_predictions,
)

print("First five predictions:", np.round(additive_forest_predictions[:5], 2))
print(f"Mean baseline MAE: {additive_baseline_mae:.2f} percentage points")
print(f"Linear regression MAE: {additive_linear_mae:.2f} percentage points")
print(f"Random forest MAE: {additive_forest_mae:.2f} percentage points")

First five predictions: [84.48  1.77 63.01 46.82  6.84]
Mean baseline MAE: 23.36 percentage points
Linear regression MAE: 13.73 percentage points
Random forest MAE: 12.20 percentage points


In [8]:
from xgboost import XGBRegressor

random_xgb_model = Pipeline([
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False,
    )),
    ("regressor", XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        objective="reg:squarederror",
        tree_method="hist",
        random_state=42,
        n_jobs=-1,
    )),
])

random_xgb_model.fit(
    random_train[component_columns],
    random_train["Output"],
)

random_xgb_predictions = random_xgb_model.predict(
    random_test[component_columns]
)

random_xgb_mae = mean_absolute_error(
    random_test["Output"],
    random_xgb_predictions,
)

print("First five predictions:", np.round(random_xgb_predictions[:5], 2))
print(f"Random forest MAE: {random_forest_mae:.2f} percentage points")
print(f"XGBoost MAE: {random_xgb_mae:.2f} percentage points")

First five predictions: [24.75 11.8   8.36  3.82 35.52]
Random forest MAE: 6.03 percentage points
XGBoost MAE: 8.87 percentage points


In [9]:
from sklearn.base import clone

additive_xgb_model = clone(random_xgb_model)

additive_xgb_model.fit(
    additive_train[component_columns],
    additive_train["Output"],
)

additive_xgb_predictions = additive_xgb_model.predict(
    additive_test[component_columns]
)

additive_xgb_mae = mean_absolute_error(
    additive_test["Output"],
    additive_xgb_predictions,
)

print("First five predictions:", np.round(additive_xgb_predictions[:5], 2))
print(f"Mean baseline MAE: {additive_baseline_mae:.2f} percentage points")
print(f"Linear regression MAE: {additive_linear_mae:.2f} percentage points")
print(f"XGBoost MAE: {additive_xgb_mae:.2f} percentage points")

First five predictions: [83.47 12.43 54.21 42.99  3.26]
Mean baseline MAE: 23.36 percentage points
Linear regression MAE: 13.73 percentage points
XGBoost MAE: 11.95 percentage points


In [10]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

morgan_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2,
    fpSize=2048,
    includeChirality=True,
)

all_smiles = pd.unique(df[component_columns].to_numpy().ravel())
fingerprint_arrays = {}

for smiles in all_smiles:
    molecule = Chem.MolFromSmiles(smiles)
    if molecule is None:
        raise ValueError(f"Invalid SMILES: {smiles}")

    fingerprint = morgan_generator.GetFingerprint(molecule)
    fingerprint_arrays[smiles] = np.array(
        list(fingerprint), dtype=np.uint8
    )

feature_blocks = [
    np.stack([fingerprint_arrays[smiles] for smiles in df[role]])
    for role in component_columns
]

X_morgan = pd.DataFrame(
    np.concatenate(feature_blocks, axis=1),
    index=df.index,
)

X_morgan_random_train = X_morgan.loc[random_train.index]
X_morgan_random_test = X_morgan.loc[random_test.index]
X_morgan_additive_train = X_morgan.loc[additive_train.index]
X_morgan_additive_test = X_morgan.loc[additive_test.index]

print("Random training features:", X_morgan_random_train.shape)
print("Random test features:", X_morgan_random_test.shape)
print("Additive training features:", X_morgan_additive_train.shape)
print("Additive test features:", X_morgan_additive_test.shape)

Random training features: (2768, 8192)
Random test features: (1187, 8192)
Additive training features: (3775, 8192)
Additive test features: (180, 8192)


In [11]:
from scipy.sparse import csr_matrix

morgan_random_linear_model = LinearRegression()

morgan_random_linear_model.fit(
    csr_matrix(X_morgan_random_train),
    random_train["Output"],
)

morgan_random_linear_predictions = morgan_random_linear_model.predict(
    csr_matrix(X_morgan_random_test)
)

morgan_random_linear_mae = mean_absolute_error(
    random_test["Output"],
    morgan_random_linear_predictions,
)

print(
    "First five predictions:",
    np.round(morgan_random_linear_predictions[:5], 2),
)
print(f"One-hot linear MAE: {random_linear_mae:.2f} percentage points")
print(f"Morgan linear MAE: {morgan_random_linear_mae:.2f} percentage points")

First five predictions: [34.7  -5.85 18.25  2.01 39.12]
One-hot linear MAE: 12.25 percentage points
Morgan linear MAE: 12.66 percentage points


In [12]:
morgan_additive_linear_model = LinearRegression()

morgan_additive_linear_model.fit(
    csr_matrix(X_morgan_additive_train),
    additive_train["Output"],
)

morgan_additive_linear_predictions = morgan_additive_linear_model.predict(
    csr_matrix(X_morgan_additive_test)
)

morgan_additive_linear_mae = mean_absolute_error(
    additive_test["Output"],
    morgan_additive_linear_predictions,
)

print(
    "First five predictions:",
    np.round(morgan_additive_linear_predictions[:5], 2),
)
print(f"One-hot linear MAE: {additive_linear_mae:.2f} percentage points")
print(f"Morgan linear MAE: {morgan_additive_linear_mae:.2f} percentage points")

First five predictions: [71.81  9.34 51.98 35.18 -7.47]
One-hot linear MAE: 13.73 percentage points
Morgan linear MAE: 13.77 percentage points


In [13]:
morgan_random_forest_model = clone(
    random_forest_model.named_steps["regressor"]
)

morgan_random_forest_model.fit(
    X_morgan_random_train,
    random_train["Output"],
)

morgan_random_forest_predictions = morgan_random_forest_model.predict(
    X_morgan_random_test
)

morgan_random_forest_mae = mean_absolute_error(
    random_test["Output"],
    morgan_random_forest_predictions,
)

print(
    "First five predictions:",
    np.round(morgan_random_forest_predictions[:5], 2),
)
print("One-hot random forest MAE: 6.03 percentage points")
print(f"Morgan random forest MAE: {morgan_random_forest_mae:.2f} percentage points")

First five predictions: [10.89 13.49  0.45  0.45 28.63]
One-hot random forest MAE: 6.03 percentage points
Morgan random forest MAE: 4.69 percentage points


In [14]:
morgan_additive_forest_model = clone(morgan_random_forest_model)

morgan_additive_forest_model.fit(
    X_morgan_additive_train,
    additive_train["Output"],
)

morgan_additive_forest_predictions = morgan_additive_forest_model.predict(
    X_morgan_additive_test
)

morgan_additive_forest_mae = mean_absolute_error(
    additive_test["Output"],
    morgan_additive_forest_predictions,
)

print(
    "First five predictions:",
    np.round(morgan_additive_forest_predictions[:5], 2),
)
print(f"Morgan random forest MAE: {morgan_additive_forest_mae:.2f} percentage points")

First five predictions: [85.39  5.45 47.42 29.51  7.8 ]
Morgan random forest MAE: 8.37 percentage points


In [15]:
morgan_random_xgb_model = clone(
    random_xgb_model.named_steps["regressor"]
)

morgan_random_xgb_model.fit(
    X_morgan_random_train,
    random_train["Output"],
)

morgan_random_xgb_predictions = morgan_random_xgb_model.predict(
    X_morgan_random_test
)

morgan_random_xgb_mae = mean_absolute_error(
    random_test["Output"],
    morgan_random_xgb_predictions,
)

print(
    "First five predictions:",
    np.round(morgan_random_xgb_predictions[:5], 2),
)
print(f"One-hot XGBoost MAE: {random_xgb_mae:.2f} percentage points")
print(f"Morgan XGBoost MAE: {morgan_random_xgb_mae:.2f} percentage points")

First five predictions: [20.09 10.96  1.32  1.29 30.3 ]
One-hot XGBoost MAE: 8.87 percentage points
Morgan XGBoost MAE: 6.86 percentage points


In [16]:
morgan_additive_xgb_model = clone(morgan_random_xgb_model)

morgan_additive_xgb_model.fit(
    X_morgan_additive_train,
    additive_train["Output"],
)

morgan_additive_xgb_predictions = morgan_additive_xgb_model.predict(
    X_morgan_additive_test
)

morgan_additive_xgb_mae = mean_absolute_error(
    additive_test["Output"],
    morgan_additive_xgb_predictions,
)

print(
    "First five predictions:",
    np.round(morgan_additive_xgb_predictions[:5], 2),
)
print(f"One-hot XGBoost MAE: {additive_xgb_mae:.2f} percentage points")
print(f"Morgan XGBoost MAE: {morgan_additive_xgb_mae:.2f} percentage points")
print(f"Morgan random forest MAE: {morgan_additive_forest_mae:.2f} percentage points")

First five predictions: [85.24  7.11 43.91 29.76  4.41]
One-hot XGBoost MAE: 11.95 percentage points
Morgan XGBoost MAE: 9.57 percentage points
Morgan random forest MAE: 8.37 percentage points


In [17]:
from sklearn.model_selection import KFold, cross_val_score

random_cv = KFold(
    n_splits=3,
    shuffle=True,
    random_state=42,
)

cv_mae = -cross_val_score(
    clone(morgan_random_forest_model),
    X_morgan_random_train,
    random_train["Output"],
    cv=random_cv,
    scoring="neg_mean_absolute_error",
    n_jobs=1,
)

print("Validation MAE by fold:", np.round(cv_mae, 2))
print(f"Mean validation MAE: {cv_mae.mean():.2f} percentage points")
print(f"Validation MAE standard deviation: {cv_mae.std():.2f}")

Validation MAE by fold: [5.74 5.28 5.69]
Mean validation MAE: 5.57 percentage points
Validation MAE standard deviation: 0.21


In [18]:
from sklearn.model_selection import GridSearchCV

parameter_grid = {
    "max_depth": [None, 15],
    "min_samples_leaf": [1, 3],
}

random_forest_search = GridSearchCV(
    estimator=clone(morgan_random_forest_model),
    param_grid=parameter_grid,
    scoring="neg_mean_absolute_error",
    cv=random_cv,
    n_jobs=1,
    refit=True,
)

random_forest_search.fit(
    X_morgan_random_train,
    random_train["Output"],
)

search_results = pd.DataFrame(
    random_forest_search.cv_results_
)

search_results["mean_validation_mae"] = -search_results["mean_test_score"]

display(
    search_results[
        ["params", "mean_validation_mae", "rank_test_score"]
    ].sort_values("rank_test_score")
)

print("Best parameters:", random_forest_search.best_params_)
print(
    f"Best mean validation MAE: "
    f"{-random_forest_search.best_score_:.2f} percentage points"
)

,params,mean_validation_mae,rank_test_score
0,"{'max_depth': None, 'min_samples_leaf': 1}",5.568016,1
2,"{'max_depth': 15, 'min_samples_leaf': 1}",5.591506,2
1,"{'max_depth': None, 'min_samples_leaf': 3}",6.262827,3
3,"{'max_depth': 15, 'min_samples_leaf': 3}",6.277461,4


Best parameters: {'max_depth': None, 'min_samples_leaf': 1}
Best mean validation MAE: 5.57 percentage points


In [19]:
from sklearn.model_selection import GroupKFold

additive_cv = GroupKFold(n_splits=3)

additive_forest_search = GridSearchCV(
    estimator=clone(morgan_additive_forest_model),
    param_grid=parameter_grid,
    scoring="neg_mean_absolute_error",
    cv=additive_cv,
    n_jobs=1,
    refit=True,
)

additive_forest_search.fit(
    X_morgan_additive_train,
    additive_train["Output"],
    groups=additive_train["Additive"],
)

additive_search_results = pd.DataFrame(
    additive_forest_search.cv_results_
)

additive_search_results["mean_validation_mae"] = (
    -additive_search_results["mean_test_score"]
)

display(
    additive_search_results[
        ["params", "mean_validation_mae", "rank_test_score"]
    ].sort_values("rank_test_score")
)

print("Best parameters:", additive_forest_search.best_params_)
print(
    f"Best grouped validation MAE: "
    f"{-additive_forest_search.best_score_:.2f} percentage points"
)

,params,mean_validation_mae,rank_test_score
2,"{'max_depth': 15, 'min_samples_leaf': 1}",9.982830,1
0,"{'max_depth': None, 'min_samples_leaf': 1}",9.986997,2
3,"{'max_depth': 15, 'min_samples_leaf': 3}",10.169553,3
1,"{'max_depth': None, 'min_samples_leaf': 3}",10.170957,4


Best parameters: {'max_depth': 15, 'min_samples_leaf': 1}
Best grouped validation MAE: 9.98 percentage points


In [20]:
selected_random_model = random_forest_search.best_estimator_
selected_additive_model = additive_forest_search.best_estimator_

selected_random_predictions = selected_random_model.predict(
    X_morgan_random_test
)
selected_additive_predictions = selected_additive_model.predict(
    X_morgan_additive_test
)

selected_random_mae = mean_absolute_error(
    random_test["Output"],
    selected_random_predictions,
)
selected_additive_mae = mean_absolute_error(
    additive_test["Output"],
    selected_additive_predictions,
)

print("Random split parameters:", random_forest_search.best_params_)
print(f"Random test MAE: {selected_random_mae:.2f} percentage points")

print("Additive split parameters:", additive_forest_search.best_params_)
print(f"Additive test MAE: {selected_additive_mae:.2f} percentage points")

Random split parameters: {'max_depth': None, 'min_samples_leaf': 1}
Random test MAE: 4.69 percentage points
Additive split parameters: {'max_depth': 15, 'min_samples_leaf': 1}
Additive test MAE: 8.40 percentage points


In [21]:
import json
import joblib
import sklearn
import rdkit
from pathlib import Path

output_dir = Path("reports/morgan_random_forest")
output_dir.mkdir(parents=True, exist_ok=True)

runs = [
    ("random", selected_random_model, random_test,
     selected_random_predictions, random_forest_search),
    ("additive", selected_additive_model, additive_test,
     selected_additive_predictions, additive_forest_search),
]

for split_name, model, test_data, predictions, search in runs:
    joblib.dump(model, output_dir / f"{split_name}_model.joblib")

    prediction_table = test_data.copy()
    prediction_table.index.name = "source_row_index"
    prediction_table["predicted_yield"] = predictions
    prediction_table["residual"] = test_data["Output"] - predictions
    prediction_table.to_csv(
        output_dir / f"{split_name}_test_predictions.csv"
    )

    pd.DataFrame(search.cv_results_).to_csv(
        output_dir / f"{split_name}_cv_results.csv",
        index=False,
    )

    config = {
        "data_file": "data/raw/Dreher_and_Doyle_input_data.xlsx",
        "sheet": "FullCV_01",
        "split_manifest": "data/splits/split_manifest.csv",
        "split": split_name,
        "roles": component_columns,
        "fingerprint": {
            "type": "Morgan",
            "radius": 2,
            "fpSize": 2048,
            "includeChirality": True,
        },
        "model_parameters": model.get_params(),
        "selection_metric": "mean validation MAE",
        "cv_folds": 3,
        "cv_type": "KFold" if split_name == "random" else "GroupKFold",
        "versions": {
            "scikit-learn": sklearn.__version__,
            "rdkit": rdkit.__version__,
            "numpy": np.__version__,
            "pandas": pd.__version__,
        },
    }

    with (output_dir / f"{split_name}_config.json").open(
        "w", encoding="utf-8"
    ) as file:
        json.dump(config, file, indent=2)

    print(f"Saved model, predictions, CV results, and config: {split_name}")

Saved model, predictions, CV results, and config: random
Saved model, predictions, CV results, and config: additive
